In [ ]:
import itertools

def generate_prompt_grid():
    """
    Génère dynamiquement toutes les combinaisons possibles de prompts.
    """
    grid = {}

    blocks = {
        "en": {
            "system_base": "You are an advanced Vision-Language Model specialized in mathematical document analysis.",
            "f_no_graphics": "Ignore drawings, graphs, charts, diagrams, or illustrations.",
            "f_typography": "Respect specific scripts, including historical handwriting, standard typewriters, Gothic (Fraktur), or calligraphic notations.",
            "f_degradations": (
                "The image might be highly degraded by noise: foxing, fungi, yellowing, paper grain, humidity marks, "
                "fading, bleed-through, feathering, pressure variations, elastic deformation, perspective distortion, "
                "binding curvature, shear, Gaussian noise, scan lines, motion blur, shadows, JPEG compression, or heavy thresholding. "
                "Ignore these artifacts."
            ),
            "f_guessing_true": "If a character or formula is partially obscured or faded, use the linguistic and mathematical context to deduce the most probable result.",
            "f_guessing_false": "CRITICAL: Do NOT attempt to guess obscured or missing parts. If a character is unreadable, replace it strictly with '[?]'.",
            "f_cot_true": "Respond strictly in JSON format with keys: 'thought' (your step-by-step analysis), 'transcription' (the valid LaTeX syntax), and 'confidence_score' (float between 0.0 and 1.0).",
            "f_cot_false": "Respond strictly in JSON format with keys: 'transcription' (the valid LaTeX syntax) and 'confidence_score' (float between 0.0 and 1.0). Do not include a thought process.",
            "user": "Analyze this document image and provide the JSON output according to the system instructions."
        },
        "fr": {
            "system_base": "Vous êtes un modèle de vision-langage avancé spécialisé dans l'analyse de documents mathématiques.",
            "f_no_graphics": "Ne pas transcrire ni décrire les dessins, graphiques, diagrammes ou illustrations.",
            "f_typography": "Respectez les écritures spécifiques (écriture manuscrite historique, machine à écrire, gothique/Fraktur, calligraphies).",
            "f_degradations": (
                "L'image peut être fortement dégradée : foxing, moisissures, jaunissement, grain du papier, traces d'humidité, "
                "décoloration, transparence (bleed-through), encrage irrégulier, variations de pression, déformation élastique, "
                "distorsion de perspective, courbure de reliure, cisaillement, bruit gaussien, lignes de balayage, flou de bougé, "
                "ombres, compression JPEG ou binarisation agressive. Ignorez ces artefacts."
            ),
            "f_guessing_true": "Si une partie est occultée ou effacée, utilisez le contexte linguistique et mathématique pour deviner le résultat le plus probable.",
            "f_guessing_false": "CRITICAL : Ne tentez PAS de deviner les parties occultées ou manquantes. Si un caractère est illisible, remplacez-le strictement par '[?]'.",
            "f_cot_true": "Répondez strictement au format JSON avec les clés : 'thought' (analyse étape par étape), 'transcription' (syntaxe LaTeX valide), et 'confidence_score' (float entre 0.0 et 1.0).",
            "f_cot_false": "Répondez strictement au format JSON avec les clés : 'transcription' (syntaxe LaTeX valide) et 'confidence_score' (float entre 0.0 et 1.0). N'incluez pas de réflexion.",
            "user": "Analysez l'image de ce document et fournissez la sortie JSON selon les instructions système."
        }
    }

    languages = ["en", "fr"]
    keep_graphics_rules = [False, False]
    keep_typography_rules = [False, False]
    keep_degradations_list = [True, True]
    guessing_modes = ["guess", "strict"]  # guess = deviner, strict 
    cot_modes = ["no_cot", "no_cot"]    # avec ou sans chaîne de pensée

    all_combinations = itertools.product(
        languages, 
        keep_graphics_rules, 
        keep_typography_rules, 
        keep_degradations_list, 
        guessing_modes, 
        cot_modes
    )

    for lang, graph, typo, deg, guess, cot in all_combinations:
        # Construction de l'identifiant unique du prompt (ex: "en_graph1_typo0_deg1_guess_nocot")
        prompt_id = f"{lang}_graph{int(graph)}_typo{int(typo)}_deg{int(deg)}_{guess}_{cot}"
        
        # Sélection du dictionnaire de langue
        b = blocks[lang]
        
        # Construction dynamique du System Prompt
        system_segments = [b["system_base"], "\n### CONSTRAINTS:"]
        
        if graph:
            system_segments.append(f"- {b['f_no_graphics']}")
        if typo:
            system_segments.append(f"- {b['f_typography']}")
        if deg:
            system_segments.append(f"- {b['f_degradations']}")
            
        system_segments.append(f"- {b['f_guessing_true'] if guess == 'guess' else b['f_guessing_false']}")
        
        system_segments.append("\n### OUTPUT FORMAT:")
        system_segments.append(b["f_cot_true"] if cot == "with_cot" else b["f_cot_false"])
        
        # Assemblage final
        grid[prompt_id] = {
            "metadata": {
                "language": lang,
                "has_graphics_rule": graph,
                "has_typography_rule": typo,
                "has_degradation_list": deg,
                "guessing_mode": guess,
                "cot_mode": cot
            },
            "system": "\n".join(system_segments),
            "user": b["user"]
        }
        
    return grid

if __name__ == "__main__":
    prompt_dataset = generate_prompt_grid()
    
    print(f"Nombre total de configurations générées : {len(prompt_dataset)}")
    print("Clés générées :")
    print(list(prompt_dataset.keys()))
    print("-" * 60)

Nombre total de configurations générées : 4
Clés générées :
['en_graph0_typo0_deg1_guess_no_cot', 'en_graph0_typo0_deg1_strict_no_cot', 'fr_graph0_typo0_deg1_guess_no_cot', 'fr_graph0_typo0_deg1_strict_no_cot']
------------------------------------------------------------
